In [102]:
import sys
from awsglue.transforms import *
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from awsglue.context import GlueContext
from awsglue.job import Job
from pyspark.sql.functions import col, upper, current_date
from awsglue.dynamicframe import DynamicFrame

sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session

sc._jsc.hadoopConfiguration().set('fs.s3a.endpoint', 'http://minio:9000')
sc._jsc.hadoopConfiguration().set('fs.s3a.access.key', 'minioadmin')
sc._jsc.hadoopConfiguration().set('fs.s3a.secret.key', 'minioadmin')
sc._jsc.hadoopConfiguration().set('fs.s3a.path.style.access', 'true')
sc._jsc.hadoopConfiguration().set('fs.s3a.impl', 'org.apache.hadoop.fs.s3a.S3AFileSystem')
sc._jsc.hadoopConfiguration().set('fs.s3a.aws.credentials.provider', 'org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider')
sc._jsc.hadoopConfiguration().set('fs.s3a.connection.ssl.enabled', 'false')

print("spark session created")

## Create dynamic frame from  s3 files

In [ ]:
datasource = glueContext.create_dynamic_frame.from_options(
    connection_type="s3",
    connection_options={"paths": [input_path]},
    format="csv",
    format_options={"withHeader": True, "separator": ","}
)
datasource.show(5)

## Write dynamic frame to s3 file

In [ ]:
glueContext.write_dynamic_frame.from_options(
    frame = datasource,
    connection_type = "s3",
    connection_options = {"path": "s3://sanjay-de-bucket-2026/processed/orders_parquet/"}, 
    format = "parquet"
)

# Create Dynamic Frame from Catalog

In [ ]:
data_catalog_source_orders= glueContext.create_dynamic_frame.from_catalog(
    database='ecommerce_db',
    table_name='orders'
)

# [print(i) for i in dir(data_catalog_source_orders) if not i.startswith('_')]

df1= data_catalog_source_orders.toDF()
df1.cache()

In [ ]:
from awsglue.dynamicframe import DynamicFrame

dyf2= DynamicFrame.fromDF(df1, glueContext, "test")
dyf2.count()

# Write Dynamic Frame from Catalog

In [ ]:
[print(m) for m in dir(glueContext) if m.startsWith("w")]

In [ ]:

#Here I am using same source table , create a different table for DEMO 
glueContext.write_dynamic_frame.from_catalog(
    frame = dyf2,
    database = 'ecommerce_db',
    table_name = 'orders',
    transformation_ctx = 'write_to_catalog')

I have noticed that dynamic frame have methods to write the dynamic frame which makes sense but it also have methods to write dataframes ?
Why would one use dynamic frame method to write a dataframe when we can directly use native write method of apache spark

So in short — if you need job bookmarks or Glue Catalog updates, use glueContext.write_dataframe. Otherwise just use native df.write which is simpler and more flexible.

In practice most people just do:

# transform using DataFrame API
df_transformed = dyf.toDF()
# ... transformations ...

# convert back and write via Glue to get bookmarks/catalog update
glueContext.write_dynamic_frame.from_options(
    frame=DynamicFrame.fromDF(df_transformed, glueContext, "output"),
    ...
)


So glueContext.write_dataframe is rarely used — it's just there for convenience so you don't have to convert to DynamicFrame first.